In [98]:
import pandas as pd
import numpy as np
from collections import deque
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense, InputLayer, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import random
import time 
import tensorflow as tf
from Data_Preprocessing import Data_Preprocessing
from Calculate_Returns import Calculate_Returns
from Triple_Barrier_Labelel import Triple_Barrier_Labelel
from Model_Train import Model_Train
import torch
import os
import gc
from Association_Rule_Mining import create_tweet_dataset, create_continous_dataset, create_triple_barrier_labeling, create_categorize_dataset
from math import sqrt
from scipy.stats import t as t_dist
import matplotlib.pyplot as plt

DEBUG = False

dp = Data_Preprocessing()
mt = Model_Train('', '')
encoder = OneHotEncoder(sparse=False)
scaler = MinMaxScaler()
tbl = Triple_Barrier_Labelel()

MEMORY_LENGTH = 100
BATCH_SIZE = 64
MODEL_DESIGN = "64/64"
LEARNING_RATE = 0.0005
FEE = 0.01
INITIAL_CASH = 100000
TARGET_UPDATE = 20
EPISODES = 50
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.95
GAMMA = 0.95
EPSILON = 1.0
TWEETS_RANDOM_SAMPLE = True
TWEETS_ENGAGEMENT_POSTS = False
TWEETS_DAILY_SAMPLE_SIZE = 50
TWEETS_RANDOM_SEED = 42
TEST_REPEATS = 5

STARTING_DATE = "2018-01-01"
ENDING_DATE ="2021-01-01"

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
def evaluate_model(original_data, scaled_data, mod, log = False, market = "BTC"):
    print("Evaluating...")
    returns = Calculate_Returns(INITIAL_CASH, FEE, pd.Series([x[0] for x in original_data]), pd.Series(dtype="float64"), pd.Series(dtype="float64"), pd.Series(dtype="float64"))
    profits = []
    action = None

    for step, el in enumerate(original_data[:-1]):
        print(step)
        offset = step
        q_values = mod.predict(np.array(scaled_data[offset]).reshape(1,-1), verbose=0)
        action = np.argmax(q_values[0])
        offset += 1
        if log:
            print(f"Step: {offset}/{len(original_data[:-1])}, Prev Price: {original_data[offset - 1][0]}, Selected Action: {action}")
        delta = ((original_data[offset][0] - original_data[offset - 1][0]) / original_data[offset - 1][0])        
        if action == 0: #Hold
            reward = -0.01
        elif action == 1: #Short
            reward = -delta
        else: #Long
            reward = delta
        returns.perform_action(action, offset)
        if log:
            print(f"Next Price: {original_data[offset][0]}, Reward: {reward}\n")
        profits.append(reward)
    positive_rewards = (len([p for p in profits if p > 0])/len(profits))*100
    if len(returns.records):
        wins = (len([e['PnL'] for e in returns.records if e['PnL'] > 0])/len(returns.records))*100
        sum_pnl = sum([e['PnL'] for e in returns.records])
        days = 365 if market == "BTC" else 252
        sharpe_ratio = returns.sharpe(days)

    if log and len(returns.records):
        print(f"Positive Rewards {positive_rewards}")
        print(f"Wins {wins}")
        print(f"Sum PnL {sum_pnl}")
        print(f"Sharpe Ratio {sharpe_ratio}")
        
    return {
        "Positive Rewards": positive_rewards,
        "Wins": wins if len(returns.records) else 0,
        "Sum PnL": sum_pnl if len(returns.records) else 0,
        "Sharpe Ratio": sharpe_ratio if len(returns.records) else 0
    }

In [ ]:
class Agent():
    def __init__(self, action_size, state_size, gamma, epsilon, epsilon_min, epsilon_decay):
        self.action_size = action_size
        self.state_size = state_size
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.memory = deque(maxlen=MEMORY_LENGTH)
        self.model = self.create_model()
        self.target_model = clone_model(self.model)
        self.optimizer = Adam(learning_rate=LEARNING_RATE) 
        self.step_counter = 0
        
    def generate_model(self, state_size, action_size):
        layers = MODEL_DESIGN.split("/")
        model = Sequential()
        state_size = state_size
        model.add(InputLayer(input_shape=(state_size,), name="InputLayer"))
        for index, units in enumerate(layers):
            model.add(Dense(units=units, activation="relu", name=f"HiddenLayer{index}"))
            model.add(Dropout(0.1))
        model.add(Dense(units=action_size, activation='linear', name="OutputLayer"))
        return model

    def create_model(self):
        return self.generate_model(self.state_size, self.action_size)

    def act(self, state):
        if random.uniform(0,1) < self.epsilon:
            rand_action = random.randrange(self.action_size)
            return rand_action
        state = np.array(state).reshape(1, -1)
        q_values = self.model(np.array(state, dtype=np.float32), training=False).numpy()
        return np.argmax(q_values[0])
    
    def remember(self, state, action, reward, new_state, done):
        self.memory.append((state, action, reward, new_state, done))
        
    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())
    
    def replay(self):
        if len(self.memory) < BATCH_SIZE:
            return
        minibatch = random.sample(self.memory, BATCH_SIZE)
        states, actions, rewards, new_states, done = zip(*minibatch)
        states = np.array(states, dtype=np.float32)
        new_states = np.array(new_states, dtype=np.float32)
        actions = np.array(actions, dtype=np.int32)
        rewards = np.array(rewards, dtype=np.float32)
        done = np.array(done, dtype=np.float32)

        best_action_indices = np.argmax(self.model(new_states, training=False).numpy(), axis=1)
        target = rewards + (1 - done) * self.gamma * self.target_model(new_states, training=False).numpy()[np.arange(BATCH_SIZE), best_action_indices]
        with tf.GradientTape() as tape:
            current_Q_values = self.model([states], training=True)
            action_mask = tf.one_hot(actions, current_Q_values.shape[1])
            predicted_Q_values = tf.reduce_sum(current_Q_values * action_mask, axis=1)
            loss = tf.keras.losses.MeanSquaredError()(target, predicted_Q_values)
        gradients = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients,self.model.trainable_variables))
        self.step_counter += 1
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        if self.step_counter % TARGET_UPDATE == 0:
            self.step_counter = 0
            self.update_target_model()
        del states, actions, rewards, new_states, done, minibatch
        gc.collect()

In [ ]:
class Environment():
    def __init__(self, data, data_scaled):
        self.action_size = 3
        self.state_size = np.array(data_scaled).shape[1]
        self.data = data
        self.data_scaled = data_scaled
        self.offset = 0
        self.steps = len(data)
        self.returns = Calculate_Returns(INITIAL_CASH, FEE, pd.Series([x[0] for x in data]), pd.Series(dtype="float64"), pd.Series(dtype="float64"), pd.Series(dtype="float64"))
    
    def _append_action_and_position_to_state(self, state, action):
        one_hot_action = np.zeros(self.action_size)
        if action is not None:
            one_hot_action[action] = 1
        position_one_hot = np.zeros(3)
        position_index = int(self.returns.context)
        position_one_hot[position_index] = 1
        return np.concatenate([state, one_hot_action, position_one_hot])
    
    def step(self, action):
        self.offset = self.offset + 1
        new_state = self.data_scaled[self.offset]
        delta = ((self.data[self.offset][0] - self.data[self.offset - 1][0]) / self.data[self.offset - 1][0])
        if action == 0: #Hold
            reward = -0.01
        elif action == 1: #Short
            reward = -delta
        else: #Long
            reward = delta
        self.returns.perform_action(action, self.offset)
        done = self.offset == len(self.data) - 1
        return new_state, reward, done
    
    def reset(self):
        self.offset = 0
        self.returns = Calculate_Returns(INITIAL_CASH, FEE, pd.Series([x[0] for x in self.data]), pd.Series(dtype="float64"), pd.Series(dtype="float64"), pd.Series(dtype="float64"))
        state_only = self.data_scaled[self.offset]
        return state_only

In [ ]:
class DQNAlgorithm():
    def __init__(self, data, data_scaled, episodes, gamma, epsilon, epsilon_min, epsilon_decay):
        self.episodes = episodes
        self.data = data
        self.data_scaled = data_scaled
        self.env = Environment(data, data_scaled)
        self.steps = self.env.steps
        self.agent = Agent(self.env.action_size, self.env.state_size, gamma, epsilon, epsilon_min, epsilon_decay)
        self.best_model_win_rate = 0
        encoder.fit([[0], [1], [2]])
        self.best_models = []
        
    def debug(self):
        print("Debugging...")
        state = self.env.reset()
        profits = []
        for step in range(self.steps):
            action = self.agent.act(state)
            new_state, reward, done = self.env.step(action)
            print(f"Step: {step}/{self.steps}, Prev Price: {self.data[step][0]}, Previous Context: {self.env.returns.context}, Selected Action: {action}")
            print(f"Next Price: {self.data[step + 1][0]}, Reward: {reward}\n")
            self.agent.remember(state, action, reward, new_state, done)
            profits.append(reward)
            if done == True:
                pos_rewards = (len([p for p in profits if p > 0])/len(profits))*100
                if pos_rewards > self.best_model_win_rate:
                    self.best_model_win_rate = pos_rewards
                    self.best_model = self.agent.model
                print(f"Positive Rewards: {pos_rewards} %")
                if len(self.env.returns.records):
                    print(f"Wins: {(len([e['PnL'] for e in self.env.returns.records if e['PnL'] > 0])/len(self.env.returns.records))*100} %")
                    print(f"Sum PnL: {sum([e['PnL'] for e in self.env.returns.records])}")
                print(f"Sharpe Ratio: {self.env.returns.sharpe()}")
                break
            self.agent.replay()
            state = new_state

    def run(self):
        print("Training...")
        for episode in range(self.episodes):
            state = self.env.reset()
            actions = 0
            start_time = time.time()
            for step in range(self.steps):
                actions+=1
                action = self.agent.act(state)
                new_state, reward, done = self.env.step(action)
                self.agent.remember(state, action, reward, new_state, done)
                if done == True:
                    new_model = clone_model(self.agent.model)
                    new_model.set_weights(self.agent.model.get_weights())
                    self.best_models.append(new_model)
                    print(f"Episode {episode + 1}/{self.episodes}")
                    end_time = time.time()
                    print(f"Elapsed time: {end_time - start_time}\n")
                    break
                self.agent.replay()
                state = new_state
        print("Cleaning memory...")
        gc.collect()
        tf.keras.backend.clear_session()
        return self.best_models

In [100]:
def calc_ema_volatility(df, span=7):
    prev_day_start = df.close.index.searchsorted(df.close.index - pd.Timedelta(days=1))
    prev_day_start = prev_day_start[prev_day_start > 0]
    prev_day_start = pd.Series(df.close.index[prev_day_start - 1], index=df.close.index[df.close.shape[0] - prev_day_start.shape[0]:])
    daily_returns = df.close.loc[prev_day_start.index] / df.close.loc[prev_day_start.values].values - 1
    vol = daily_returns.ewm(span=span).std()

    rolling_mean = vol.rolling(window=span).mean()
    return rolling_mean.mean()

def calc_max_drawdown(df):
    df['return'] = df.close.pct_change()
    df['cum'] = (1 + df['return']).cumprod()
    running_max = df['cum'].cummax()
    mdd = ((df['cum'] - running_max) / running_max).min()
    return mdd

def compare_drawdowns_and_volatilities():
    continous_df_btc = create_continous_dataset('BTC-USD', starting_date=STARTING_DATE, ending_date=ENDING_DATE)
    print(f"BTC-USD Volatility: {calc_ema_volatility(continous_df_btc)}")
    print(f"BTC-USD Max-Drawdown: {calc_max_drawdown(continous_df_btc)}")
    continous_df_msft = create_continous_dataset('MSFT', starting_date=STARTING_DATE, ending_date=ENDING_DATE)
    print(f"MSFT Volatility: {calc_ema_volatility(continous_df_msft)}")
    print(f"MSFT Max-Drawdown: {calc_max_drawdown(continous_df_msft)}")
    continous_df_amzn = create_continous_dataset('AMZN', starting_date=STARTING_DATE, ending_date=ENDING_DATE)
    print(f"AMZN Volatility: {calc_ema_volatility(continous_df_amzn)}")
    print(f"AMZN Max-Drawdown: {calc_max_drawdown(continous_df_amzn)}")
compare_drawdowns_and_volatilities()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

BTC-USD Volatility: 0.040751516678197465
BTC-USD Max-Drawdown: -0.7203250781217104
MSFT Volatility: 0.019771018005093698
MSFT Max-Drawdown: -0.2803928291661194
AMZN Volatility: 0.022210959968038332
AMZN Max-Drawdown: -0.34103783046300634


In [101]:
def evaluate_buy_and_hold(market_data):
    df = market_data.copy()
    df["Return"] = df["close"].pct_change()
    df = df.dropna()

    # Buy once, hold all the way
    cumulative_return = (1 + df["Return"]).prod() - 1

    # Win rate = fraction of days with positive daily return
    win_rate = (df["Return"] > 0).mean() * 100

    return cumulative_return * 100, win_rate

def evaluate_sma_crossover(market_data, short_window=10, long_window=30):
    df = market_data.copy()
    df["SMA_Short"] = df["close"].rolling(window=short_window).mean()
    df["SMA_Long"] = df["close"].rolling(window=long_window).mean()
    df = df.dropna()

    # Position: 1 = long, -1 = short (you can also use 0 for flat if you prefer)
    df["Position"] = np.where(df["SMA_Short"] > df["SMA_Long"], 1, -1)

    # Daily returns
    df["Return"] = df["close"].pct_change()
    df["Strategy_Return"] = df["Position"].shift(1) * df["Return"]

    # Metrics
    cumulative_return = (1 + df["Strategy_Return"]).prod() - 1
    win_rate = (df["Strategy_Return"] > 0).mean() * 100

    return cumulative_return * 100, win_rate

def optimize_crossover(dataset):
    shorts = list(range(5,20))
    longs = list(range(20,50))

    best = 0
    short_wind = 0
    long_wind = 0

    for s in shorts:
        for l in longs:
            win_rate = evaluate_sma_crossover(pd.DataFrame(dataset, columns=["close"]), s, l)[1]
            if win_rate > best:
                best = win_rate
                short_wind = s
                long_wind = l
    print(best, short_wind, long_wind)


btc_df = create_continous_dataset('BTC-USD', starting_date=STARTING_DATE, ending_date=ENDING_DATE)
optimize_crossover(btc_df)
print(evaluate_buy_and_hold(btc_df))

amzn_df = create_continous_dataset('AMZN', starting_date=STARTING_DATE, ending_date=ENDING_DATE)
optimize_crossover(amzn_df)
print(evaluate_buy_and_hold(amzn_df))

msft_df = create_continous_dataset('MSFT', starting_date=STARTING_DATE, ending_date=ENDING_DATE)
optimize_crossover(msft_df)
print(evaluate_buy_and_hold(msft_df))

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

54.722492697176236 13 21
(158.36032184204387, 53.25047801147228)



[*********************100%***********************]  1 of 1 completed

53.89133627019089 10 27
(104.70960444348219, 54.39093484419264)


56.477438136826784 9 21
(146.09853656081842, 56.798866855524075)


In [ ]:
cases = pd.read_csv("TestCases/DDQN_Test_Cases.csv")
for test_id in range(20, len(cases)):
    params = cases["Parameters"][test_id]
    market = cases["Market"][test_id]
    name = cases["TestCaseName"][test_id]
    
    continous_df = create_continous_dataset(market, starting_date=STARTING_DATE, ending_date=ENDING_DATE)
    continous_df_with_tbl = create_triple_barrier_labeling(continous_df)
    tweets_df = create_tweet_dataset("Datasets/investing_classified_sentiments" if market != "BTC-USD" else "Datasets/btc_classified_sentiments") #"Datasets/investing_classified_sentiments"
    merged_df = pd.concat([continous_df_with_tbl, tweets_df], axis=1).dropna()
    # extended = merged_df #extend_time_columns(merged_df, skip_cols=["signals"], t=7)
    categorized_for_trade_profitable = create_categorize_dataset(merged_df, suffix_vals=['bearish', 'Bearish', 'bullish', 'Bullish', '1.0', '1', '0.0', '0', '-1', '-1.0'], skip_cols=["next_day_label", "signals", "previous_label"])
    merged_df = pd.concat([merged_df, categorized_for_trade_profitable], axis=1).dropna()

    sel_param = ["close"] + [el.strip() for el in params.split(",")]
    
    existing_cols = [col for col in sel_param if col in merged_df.columns]
    merged_data = merged_df[existing_cols]
    
    merged_data = merged_data.values.tolist()
    train_end = int(len(merged_data) * 0.8)
    train_data = merged_data[:train_end]
    test_data  = merged_data[train_end:]
    train_x = np.array(train_data)[:,1:]
    train_y = np.array(train_data)[:,:1]
    test_x = np.array(test_data)[:,1:]
    test_y = np.array(test_data)[:,:1]
    scaler.fit(train_x)
    train_scaled = scaler.transform(train_x).tolist()
    test_scaled = scaler.transform(test_x).tolist()
    
    test_results_df = []
    
    for rep in range(0, TEST_REPEATS):

        dqn = DQNAlgorithm(data=train_y, data_scaled=train_scaled, episodes=EPISODES, gamma=GAMMA, epsilon=EPSILON, epsilon_min=EPSILON_MIN, epsilon_decay = EPSILON_DECAY)
        if DEBUG == True:
            dqn.debug(train_data)
            break
        else:
            best_models = dqn.run()

        test_results = {
            "Positive Rewards": -np.inf,
            "Wins": -np.inf,
            "Sum PnL": -np.inf,
            "Sharpe Ratio": -np.inf
        }
        
        for index, model in enumerate(best_models):
            score = evaluate_model(test_y, test_scaled, model, False, market)
            if score["Positive Rewards"] > test_results["Positive Rewards"]:
                test_results = score

        test_results_df.append(test_results)

    test_results_df = pd.DataFrame(test_results_df)
    columns = ["TestCaseName", "Test Repeats", "Train Episodes", "Market", "Parameters", "Positive Rewards", "Wins", "Sum PnL", "Sharpe Ratio"]
    test_conc = [name, TEST_REPEATS, EPISODES, market, params, f"Mean: {test_results_df['Positive Rewards'].mean()}, Std: {test_results_df['Positive Rewards'].std()}", f"Mean: {test_results_df['Wins'].mean()}, Std: {test_results_df['Wins'].std()}", f"Mean: {pd.to_numeric(test_results_df['Sum PnL'], errors='coerce').mean()}, Std: {pd.to_numeric(test_results_df['Sum PnL'], errors='coerce').std()}", f"Mean: {test_results_df['Sharpe Ratio'].mean()}, Std: {test_results_df['Sharpe Ratio'].std()}"]
    pd.DataFrame([test_conc], columns=columns).to_csv(f"TestResults/DDQN_Test_Results.csv", mode="a", index=False, header=not os.path.exists("TestResults/DDQN_Test_Results.csv"))


In [ ]:
cases = pd.read_csv("TestResults/DDQN_Test_Results.csv")

In [ ]:
def welch_t_test(mean1, std1, n1, mean2, std2, n2):
    # Welch t-statistic
    numerator = mean1 - mean2
    denominator = sqrt((std1**2)/n1 + (std2**2)/n2)
    t_stat = numerator / denominator

    # Degrees of freedom
    df_num = ((std1**2)/n1 + (std2**2)/n2)**2
    df_den = ((std1**2)**2) / (n1**2 * (n1-1)) + ((std2**2)**2) / (n2**2 * (n2-1))
    df = df_num / df_den

    # Two-tailed p-value
    p_value = 2 * (1 - t_dist.cdf(abs(t_stat), df))

    return t_stat, p_value

In [ ]:
#2018-2019
#AMZN
print("***AMZN***")
print(welch_t_test(63.64, 3.21, 5, 66.21, 6.22, 5)) #Sentiment Only
print(welch_t_test(64.55, 2.03, 5, 66.21, 6.22, 5)) #Technical Only
print(welch_t_test(62.42, 3.28, 5, 66.21, 6.22, 5)) #TP High lift Only
print(welch_t_test(65.81, 1.33, 5, 66.21, 6.22, 5)) #TP Low lift Only
print(welch_t_test(57.45, 6.46, 5, 66.21, 6.22, 5)) #TP Zero lift Only
#BTC-USD
print("***BTC-USD***")
print(welch_t_test(54.55, 0, 5, 66.15, 0.60, 5)) #Sentiment Only
print(welch_t_test(55.35, 2.57, 5, 66.15, 0.60, 5)) #Technical Only
print(welch_t_test(57.22, 4.66, 5, 66.15, 0.60, 5)) #TP High lift Only
print(welch_t_test(51.14, 3.42, 5, 66.15, 0.60, 5)) #TP Low lift Only
print(welch_t_test(60.27, 3.92, 5, 66.15, 0.60, 5)) #All Parameters
#MSFT
print("***MSFT***")
print(welch_t_test(70, 0, 5, 75, 7.14, 5)) #Sentiment Only
print(welch_t_test(62.50, 15, 5, 75, 7.14, 5)) #Technical Only
print(welch_t_test(71.43, 3.89, 5, 75, 7.14, 5)) #TP High lift Only
print(welch_t_test(64.77, 2.27, 5, 75, 7.14, 5)) #TP Zero lift Only
print(welch_t_test(72.62, 5.99, 5, 75, 7.14, 5)) #All Parameters

In [ ]:
#2018-2021
#AMZN
print("***AMZN***")
print(welch_t_test(58.93, 0, 5, 64, 0.81, 5)) #Sentiment Only
print(welch_t_test(53.49, 2.96, 5, 64, 0.81, 5)) #Technical Only
print(welch_t_test(61.1, 0.5, 5, 64, 0.81, 5)) #TP Low lift Only
print(welch_t_test(60, 0, 5, 64, 0.81, 5)) #TP Zero lift Only
print(welch_t_test(63.09, 1.38, 5, 64, 0.81, 5)) #All Parameters
#BTC-USD
print("***BTC-USD***")
print(welch_t_test(56.32, 0.27, 5, 58.40, 2.35, 5)) #Sentiment Only
print(welch_t_test(57.05, 0.61, 5, 58.40, 2.35, 5)) #Technical Only
print(welch_t_test(56.30, 1.82, 5, 58.40, 2.35, 5)) #TP Low lift Only
print(welch_t_test(53.98, 1.98, 5, 58.40, 2.35, 5)) #TP Zero lift Only
print(welch_t_test(58.18, 3.35, 5, 58.40, 2.35, 5)) #All Parameters
#MSFT
print("***MSFT***")
print(welch_t_test(58.56, 0, 5, 61.47, 2.67, 5)) #Sentiment Only
print(welch_t_test(59.81, 0, 5, 61.47, 2.67, 5)) #Technical Only
print(welch_t_test(60.90, 2.43, 5, 61.47, 2.67, 5)) #TP High lift Only
print(welch_t_test(61.28, 2.19, 5, 61.47, 2.67, 5)) #TP Low lift Only
print(welch_t_test(59.62, 0.42, 5, 61.47, 2.67, 5)) #TP Zero lift Only

In [102]:
def test_and_rel_performance(mapped_df, baseline_performance):
    max_values = mapped_df.groupby("Market")["Result Mean"].max()
    max_stds = mapped_df.groupby("Market")["Result Std"].max()
    mapped_df["t-statistic"] = mapped_df.apply(lambda x: float(welch_t_test(x["Result Mean"], x["Result Std"], 5, max_values[x["Market"]], max_stds[x["Market"]], 5)[0]), axis=1)
    mapped_df["p-value"] = mapped_df.apply(lambda x: float(welch_t_test(x["Result Mean"], x["Result Std"], 5, max_values[x["Market"]], max_stds[x["Market"]], 5)[1]), axis=1)
    mapped_df["relative_result"] = (mapped_df["Result Mean"] / baseline_performance) * 100
    mapped_df["relative_std"] = mapped_df["Result Std"] * (100 / baseline_performance)
    return mapped_df

def print_full_results(market, baseline_performance):
    cases = pd.read_csv("TestResults/DDQN_Test_Results.csv")
    mapped_df = pd.DataFrame()
    pd.options.display.float_format = "{:.3f}".format
    cases = cases[~cases["TestCaseName"].str.contains("LASSO", na=False) & ~cases["TestCaseName"].str.contains("PCA", na=False)]
    cases = cases[cases["Market"] == market]
    mapped_df["Name"] = cases["TestCaseName"]
    mapped_df["Market"] = cases["Market"]
    mapped_df["Result Mean"] = cases["Positive Rewards"].apply(lambda x: float(x.split(",")[0].split(" ")[1]))
    mapped_df["Result Std"] = cases["Positive Rewards"].apply(lambda x: float(x.split(",")[1].split(" ")[2]))
    mapped_df = test_and_rel_performance(mapped_df, baseline_performance)
    return mapped_df.sort_values(by="Result Mean", ascending=False)

In [103]:
amzn_shorter_period_values = [
    {
        "Name": "AMZN_Sentiment_Only",
        "Market": "AMZN",
        "Result Mean": 63.64,
        "Result Std":  3.21
    },
    {
        "Name": "AMZN_Technical_Only",
        "Market": "AMZN",
        "Result Mean": 64.55,
        "Result Std":  2.03
    },
    {
        "Name": "AMZN_TargetProfitable_Zero_Lift",
        "Market": "AMZN",
        "Result Mean": 57.45,
        "Result Std": 6.46
    },
    {
        "Name": "AMZN_TargetProfitable_Low_Lift",
        "Market": "AMZN",
        "Result Mean": 65.81,
        "Result Std":  1.33
    },
    {
        "Name": "AMZN_TargetProfitable_High_Lift",
        "Market": "AMZN",
        "Result Mean": 62.42,
        "Result Std":  3.28
    },
    {
        "Name": "AZMN_All_Params",
        "Market": "AMZN",
        "Result Mean": 66.21,
        "Result Std":  6.22
    },
    {
        "Name": "AMZN_SMA-crossover",
        "Market": "AMZN",
        "Result Mean": 60.22,
        "Result Std":  0.00
    },
]

btcusd_shorter_period_values = [
    {
        "Name": "BTC_Sentiment_Only",
        "Market": "BTC-USD",
        "Result Mean": 54.55,
        "Result Std":  0.00
    },
    {
        "Name": "BTC_Technical_Only",
        "Market": "BTC-USD",
        "Result Mean": 55.35,
        "Result Std":  2.57
    },
    {
        "Name": "BTC_TargetProfitable_Zero_Lift",
        "Market": "BTC-USD",
        "Result Mean": 66.15,
        "Result Std": 0.60
    },
    {
        "Name": "BTC_TargetProfitable_Low_Lift",
        "Market": "BTC-USD",
        "Result Mean": 51.14,
        "Result Std":  3.42
    },
    {
        "Name": "BTC_TargetProfitable_High_Lift",
        "Market": "BTC-USD",
        "Result Mean": 57.22,
        "Result Std":  4.66
    },
    {
        "Name": "BTC_All_Params",
        "Market": "BTC-USD",
        "Result Mean": 60.27,
        "Result Std":  3.92
    },
    {
        "Name": "BTC_SMA-crossover",
        "Market": "BTC-USD",
        "Result Mean": 51.35,
        "Result Std":  0.00
    },
]

msft_shorter_period_values = [
    {
        "Name": "MSFT_Sentiment_Only",
        "Market": "MSFT",
        "Result Mean": 70.00,
        "Result Std":  0.00
    },
    {
        "Name": "MSFT_Technical_Only",
        "Market": "MSFT",
        "Result Mean": 62.50,
        "Result Std":  15
    },
    {
        "Name": "MSFT_TargetProfitable_Zero_Lift",
        "Market": "MSFT",
        "Result Mean": 64.77,
        "Result Std": 2.27
    },
    {
        "Name": "MSFT_TargetProfitable_Low_Lift",
        "Market": "MSFT",
        "Result Mean": 75,
        "Result Std":  7.14
    },
    {
        "Name": "MSFT_TargetProfitable_High_Lift",
        "Market": "MSFT",
        "Result Mean": 71.43,
        "Result Std":  3.89
    },
    {
        "Name": "MSFT_All_Params",
        "Market": "MSFT",
        "Result Mean": 72.62,
        "Result Std":  5.99
    },
    {
        "Name": "MSFT_SMA-crossover",
        "Market": "MSFT",
        "Result Mean": 55.69,
        "Result Std":  0.00
    },
]

In [104]:
#BTC-USD 2018-2019 Full Results
test_and_rel_performance(pd.DataFrame(btcusd_shorter_period_values), 50.79365079365079).sort_values(by="Result Mean", ascending=False)

,Name,Market,Result Mean,Result Std,t-statistic,p-value,relative_result,relative_std
2,BTC_TargetProfitable_Zero_Lift,BTC-USD,66.150,0.600,0.000,1.000,130.233,1.181
5,BTC_All_Params,BTC-USD,60.270,3.920,-2.159,0.064,118.657,7.718
4,BTC_TargetProfitable_High_Lift,BTC-USD,57.220,4.660,-3.030,0.016,112.652,9.174
1,BTC_Technical_Only,BTC-USD,55.350,2.570,-4.538,0.004,108.970,5.060
0,BTC_Sentiment_Only,BTC-USD,54.550,0.000,-5.566,0.005,107.395,0.000
6,BTC_SMA-crossover,BTC-USD,51.350,0.000,-7.102,0.002,101.095,0.000
3,BTC_TargetProfitable_Low_Lift,BTC-USD,51.140,3.420,-5.807,0.001,100.682,6.733


In [105]:
#BTC-USD 2018-2021 Full Results
print_full_results("BTC-USD", 53.25047801147228)

,Name,Market,Result Mean,Result Std,t-statistic,p-value,relative_result,relative_std
1,BTC_TargetProfitable_High_Lift,BTC-USD,58.405,2.352,0.000,1.000,109.680,4.417
0,BTC_All_Params,BTC-USD,58.182,3.347,-0.105,0.919,109.261,6.286
5,BTC_Technical_Only,BTC-USD,57.055,0.613,-0.887,0.422,107.145,1.152
4,BTC_Sentiment_Only,BTC-USD,56.319,0.274,-1.389,0.236,105.762,0.515
2,BTC_TargetProfitable_Low_Lift,BTC-USD,56.310,1.815,-1.231,0.263,105.745,3.409
25,BTC_SMA-crossover,BTC-USD,54.722,0.000,-2.460,0.070,102.764,0.000
3,BTC_TargetProfitable_Zero_Lift,BTC-USD,53.976,1.980,-2.547,0.041,101.362,3.718


In [106]:
#AMZN 2018-2019 Full Results
test_and_rel_performance(pd.DataFrame(amzn_shorter_period_values), 53.73134328358209).sort_values(by="Result Mean", ascending=False)

,Name,Market,Result Mean,Result Std,t-statistic,p-value,relative_result,relative_std
5,AZMN_All_Params,AMZN,66.210,6.220,0.000,1.000,123.224,11.576
3,AMZN_TargetProfitable_Low_Lift,AMZN,65.810,1.330,-0.136,0.898,122.480,2.475
1,AMZN_Technical_Only,AMZN,64.550,2.030,-0.548,0.608,120.135,3.778
0,AMZN_Sentiment_Only,AMZN,63.640,3.210,-0.797,0.457,118.441,5.974
4,AMZN_TargetProfitable_High_Lift,AMZN,62.420,3.280,-1.170,0.287,116.171,6.104
6,AMZN_SMA-crossover,AMZN,60.220,0.000,-2.073,0.107,112.076,0.000
2,AMZN_TargetProfitable_Zero_Lift,AMZN,57.450,6.460,-2.144,0.064,106.921,12.023


In [107]:
#AMZN 2018-2021 Full Results
print_full_results("AMZN", 54.39093484419264)

,Name,Market,Result Mean,Result Std,t-statistic,p-value,relative_result,relative_std
9,AMZN_TargetProfitable_High_Lift,AMZN,64.000,0.813,0.000,1.000,117.667,1.495
8,AZMN_All_Params,AMZN,63.091,1.379,-0.622,0.558,115.995,2.535
10,AMZN_TargetProfitable_Low_Lift,AMZN,61.101,0.502,-2.156,0.094,112.337,0.924
11,AMZN_TargetProfitable_Zero_Lift,AMZN,60.000,0.000,-3.017,0.039,110.312,0.000
12,AMZN_Sentiment_Only,AMZN,58.929,0.000,-3.825,0.019,108.343,0.000
24,AMZN_SMA-crossover,AMZN,53.891,0.000,-7.625,0.002,99.081,0.000
13,AMZN_Technical_Only,AMZN,53.488,2.965,-5.606,0.001,98.341,5.450


In [108]:
#MSFT 2018-2019 Full Results
test_and_rel_performance(pd.DataFrame(msft_shorter_period_values), 54.72636815920397).sort_values(by="Result Mean", ascending=False)

,Name,Market,Result Mean,Result Std,t-statistic,p-value,relative_result,relative_std
3,MSFT_TargetProfitable_Low_Lift,MSFT,75.000,7.140,0.000,1.000,137.045,13.047
5,MSFT_All_Params,MSFT,72.620,5.990,-0.329,0.755,132.697,10.945
4,MSFT_TargetProfitable_High_Lift,MSFT,71.430,3.890,-0.515,0.631,130.522,7.108
0,MSFT_Sentiment_Only,MSFT,70.000,0.000,-0.745,0.497,127.909,0.000
2,MSFT_TargetProfitable_Zero_Lift,MSFT,64.770,2.270,-1.508,0.203,118.352,4.148
1,MSFT_Technical_Only,MSFT,62.500,15.000,-1.318,0.224,114.205,27.409
6,MSFT_SMA-crossover,MSFT,55.690,0.000,-2.879,0.045,101.761,0.000


In [109]:
#MSFT 2018-2021 Full Results
print_full_results("MSFT", 56.798866855524075)

,Name,Market,Result Mean,Result Std,t-statistic,p-value,relative_result,relative_std
14,MSFT_All_Params,MSFT,61.468,2.675,0.000,1.000,108.220,4.709
16,MSFT_TargetProfitable_Low_Lift,MSFT,61.284,2.190,-0.119,0.909,107.897,3.856
15,MSFT_TargetProfitable_High_Lift,MSFT,60.901,2.434,-0.351,0.735,107.222,4.285
19,MSFT_Technical_Only,MSFT,59.813,0.000,-1.383,0.239,105.307,0.000
17,MSFT_TargetProfitable_Zero_Lift,MSFT,59.623,0.422,-1.524,0.199,104.972,0.743
18,MSFT_Sentiment_Only,MSFT,58.559,0.000,-2.432,0.072,103.098,0.000
26,MSFT_SMA-crossover,MSFT,56.477,0.000,-4.172,0.014,99.434,0.000
